# Lesson 7.2 - Multi-Agent Workflows & Orchestration (toy multi-agent demo)

## Objectives

- Implement a coordinator + worker architecture.
- Demonstrate task delegation and aggregation.
- Compare sequential and parallel execution trade-offs.


In [ ]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import Dict, List
import time


## Problem Setup

We simulate customer support orchestration with three workers:

- `triage_agent`
- `knowledge_agent`
- `response_agent`

A coordinator agent controls sequencing and final decision.


In [ ]:
TICKETS = [
    {
        "ticket_id": "T-200",
        "text": "My payment failed but account was charged.",
        "customer_tier": "gold",
    },
    {
        "ticket_id": "T-201",
        "text": "Where can I find API usage limits?",
        "customer_tier": "standard",
    },
    {
        "ticket_id": "T-202",
        "text": "I need to cancel my subscription immediately.",
        "customer_tier": "silver",
    },
]

KNOWLEDGE_BASE = {
    "billing": "If charged twice, verify transaction id and initiate reversal workflow.",
    "api": "API limits documented at /docs/rate-limits with tier-specific quotas.",
    "cancel": "Cancellation policy requires identity verification and retention offer step.",
}


In [ ]:
def triage_agent(ticket: Dict[str, str]) -> Dict[str, str]:
    text = ticket["text"].lower()
    if "charged" in text or "payment" in text:
        intent = "billing"
        priority = "high"
    elif "api" in text or "limits" in text:
        intent = "api"
        priority = "low"
    else:
        intent = "cancel"
        priority = "medium"
    return {"intent": intent, "priority": priority}


def knowledge_agent(intent: str) -> Dict[str, str]:
    return {"knowledge": KNOWLEDGE_BASE[intent]}


def response_agent(ticket: Dict[str, str], triage: Dict[str, str], kb: Dict[str, str]) -> Dict[str, str]:
    base = f"Ticket {ticket['ticket_id']} classified as {triage['intent']}. "
    answer = base + kb["knowledge"]
    if triage["priority"] == "high":
        answer += " Escalating to priority queue."
    return {"draft_response": answer}


## Coordinator Workflow (Sequential)


In [ ]:
@dataclass
class OrchestrationResult:
    ticket_id: str
    intent: str
    priority: str
    response: str


def coordinator_sequential(ticket: Dict[str, str]) -> OrchestrationResult:
    triage = triage_agent(ticket)
    kb = knowledge_agent(triage["intent"])
    response = response_agent(ticket, triage, kb)
    return OrchestrationResult(
        ticket_id=ticket["ticket_id"],
        intent=triage["intent"],
        priority=triage["priority"],
        response=response["draft_response"],
    )


seq_results = [coordinator_sequential(t) for t in TICKETS]
for r in seq_results:
    print(r)


## Coordinator Workflow (Parallel Batch)

The coordinator still controls logic, but delegates independent tickets in parallel.


In [ ]:
def run_parallel(tickets: List[Dict[str, str]]) -> List[OrchestrationResult]:
    with ThreadPoolExecutor(max_workers=3) as ex:
        futures = [ex.submit(coordinator_sequential, t) for t in tickets]
        return [f.result() for f in futures]


start = time.time()
parallel_results = run_parallel(TICKETS)
elapsed = time.time() - start
print(f"Processed {len(parallel_results)} tickets in {elapsed:.4f} seconds")


## Optional Framework Mapping

If you install `langgraph` or `crewai`, map the same structure as:

- coordinator node / manager agent
- triage, retrieval, response worker nodes
- aggregator and escalation edge

The conceptual workflow remains the same even if framework syntax differs.


## Connect to Theory

- This example uses orchestration (central coordinator), not choreography.
- Worker roles are specialized for classification, retrieval, and generation.
- Escalation logic demonstrates guardrails for high-priority paths.


## Business Case Studies & Exceptions

### Case Study A: Customer Support Operations

A fintech support stack used role-specialized agents for triage, policy retrieval, and draft generation. Time-to-first-response improved, while QA teams still reviewed high-risk billing tickets.

### Case Study B: Internal Data Team Automation

Coordinator assigned ETL checks, schema QA, and report drafting to separate agents. Incident diagnosis improved because each stage produced structured output.

### Exceptions

- If workflow has strict deterministic rules, classical workflow engines can be simpler than LLM multi-agent systems.
- If throughput is low, parallel multi-agent orchestration may add unnecessary compute cost.


## Interview Questions & Answers

1. **What is a coordinator-worker pattern?**  
   One controller decomposes and delegates tasks to specialist workers.
2. **When does multi-agent architecture add value?**  
   When specialization or parallelism materially improves quality, latency, or maintainability.
3. **What is orchestration?**  
   A central workflow controller coordinates task execution.
4. **How does it differ from choreography?**  
   Choreography is decentralized event-based interaction without one controller.
5. **What is a key failure mode?**  
   Deadlock or repeated handoff loops.
6. **How do you reduce conflicts between agents?**  
   Typed interfaces, explicit contracts, and arbitration rules.
7. **What should be logged per agent step?**  
   Inputs, outputs, tool calls, latency, and error status.
8. **Why include escalation logic?**  
   To route high-risk cases for human review.
9. **Can one model serve all agents?**  
   Yes, but role-specific prompts/models often improve cost-performance.
10. **What is the first scaling step from toy to production?**  
   Add persistent state, retries, and structured observability.
